# FRIDAY — Kronos Forecast Terminal
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ddeva6/friday/blob/main/friday_colab.ipynb)

FRIDAY is a personal NSE equity research tool that generates 5-day forecasts using the Kronos (Chronos) time-series foundation model.

This notebook runs the full pipeline end-to-end on a T4 GPU.

## Step 1: Setup & Clone
We'll clone the repository and install the necessary dependencies.

In [ ]:
!git clone https://github.com/ddeva6/friday.git
%cd friday
!pip install -q yfinance pandas pyyaml requests numpy jsonschema torch chronos-forecasting

## Step 2: Verify GPU
Ensure that you are using a T4 GPU runtime.

In [ ]:
import torch
print(f"GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only'}")
assert torch.cuda.is_available(), "Switch runtime to T4 GPU: Runtime → Change runtime type → T4 GPU"

## Step 3: Configure
Set the date range for data fetching and forecast parameters.

In [ ]:
import yaml
from datetime import datetime, timedelta

end = datetime.now().strftime('%Y-%m-%d')
start = (datetime.now() - timedelta(days=365)).strftime('%Y-%m-%d')

config = {
    'database': {'path': 'friday.db'},
    'dates': {'start': start, 'end': end},
    'forecast': {
        'lookback': 180,
        'horizon': 5,
        'sample_count': 30,
        'model': 'amazon/chronos-t5-base'
    }
}
with open('config.yaml', 'w') as f:
    yaml.dump(config, f)
print(f"Config: {start} → {end}")

## Phase 1: Fetch EOD Data
Download historical OHLCV data for Nifty 50 and its constituents from Yahoo Finance.

In [ ]:
from fetch_eod import run_fetcher
run_fetcher()

## Phase 2: Run Kronos/Chronos Forecasts
Generate 5-day probabilistic forecasts using the Chronos foundation model.

In [ ]:
import time
from forecast_batch import run_forecast_batch

torch.cuda.empty_cache()
torch.cuda.reset_peak_memory_stats()
start_time = time.time()

run_forecast_batch()

end_time = time.time()
duration = end_time - start_time
peak_vram = torch.cuda.max_memory_allocated() / (1024**3)

print(f"Forecast batch completed in {duration:.2f}s")
print(f"Peak VRAM usage: {peak_vram:.2f} GB")

## Phase 3: Export JSON
Export the data to JSON files for the frontend dashboard.

In [ ]:
from export_json import export_data
export_data()

## Verification: Run Tests
Run unit and integration tests to ensure data integrity and forecast quality.

In [ ]:
print("Running unit tests...")
!python -m pytest test_phase1.py test_phase2.py -v -m "not integration"
print("\nRunning integration tests (GPU)...")
!python -m pytest test_phase2.py -v -m integration

## Serve the Dashboard
Access the interactive terminal via the Colab proxy URL.

In [ ]:
import subprocess, time
from google.colab.output import eval_js

# Start a simple HTTP server
proc = subprocess.Popen(['python', '-m', 'http.server', '8080'])
time.sleep(2)

# Get the public URL
url = eval_js("google.colab.kernel.proxyPort(8080)")
dashboard_url = f"{url}friday-forecast-terminal.html"
print(f"Dashboard URL: {dashboard_url}")

## Summary Report

In [ ]:
import sqlite3
conn = sqlite3.connect('friday.db')
c = conn.cursor()

c.execute("SELECT COUNT(*) FROM instruments")
total_instruments = c.fetchone()[0]

c.execute("SELECT COUNT(*) FROM forecasts")
total_forecasts = c.fetchone()[0]

conn.close()

print(f"--- FRIDAY Pipeline Summary ---")
print(f"Total instruments fetched:   {total_instruments}")
print(f"Total forecasts generated:   {total_forecasts}")
print(f"Time taken for batch:        {duration:.2f}s")
print(f"Peak VRAM usage:             {peak_vram:.2f} GB")
print(f"Dashboard URL:               {dashboard_url}")
print(f"-------------------------------")